# Multi-hop GraphRAG (Ontology-Constrained) + Retrieval Metrics


In [19]:
from pathlib import Path
import os
BASE_DIR=Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф')
QUESTIONS_FILE=BASE_DIR/'annotaion_q_graph_questions.txt'
ANNOTATIONS_DIR=BASE_DIR/'annotaion_q_graph'
OUTPUT_DIR=BASE_DIR/'retrieval_outputs'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
SELECTED_QNUMS=[1]
NEO4J_URI=os.getenv('NEO4J_URI','bolt://localhost:7689'); NEO4J_USER=os.getenv('NEO4J_USER','neo4j'); NEO4J_PASSWORD=os.getenv('NEO4J_PASSWORD','12345678')
MAX_HOPS=3; MAX_PATHS_PER_TEMPLATE=2000; TOP_K_PATHS= 2500; MAX_BFS_PATHS=2000
CSV_OUT=OUTPUT_DIR/'graphrag_triplet_retrieval_metrics.csv'; DETAIL_CSV_OUT=OUTPUT_DIR/'graphrag_triplet_retrieval_details.csv'
print(CSV_OUT)



/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/graphrag_triplet_retrieval_metrics.csv


In [20]:
import re,json
from collections import defaultdict
from dataclasses import dataclass
import pandas as pd
try:
    from neo4j import GraphDatabase
except Exception as e:
    GraphDatabase=None
    print('neo4j import error:',e)

def safe_load_json(path):
    try:return json.loads(path.read_text(encoding='utf-8'))
    except Exception:return None

def canonicalize_key(x):
    return '' if x is None else str(x).strip().lower()

def triple_to_tuple(tr):
    s=tr.get('s',{}) or {}; o=tr.get('o',{}) or {}
    return (canonicalize_key(s.get('key')),canonicalize_key(tr.get('p')),canonicalize_key(o.get('key')),canonicalize_key(s.get('label')),canonicalize_key(o.get('label')))

def parse_questions(path):
    out={}
    for line in path.read_text(encoding='utf-8').splitlines():
        m=re.match(r'^(\d+)\s+(.+)$',line.strip())
        if m: out[int(m.group(1))]=m.group(2).strip()
    return out


In [21]:
questions=parse_questions(QUESTIONS_FILE)
missing=[q for q in SELECTED_QNUMS if q not in questions]
if missing: raise ValueError(f'Missing qnums: {missing}')

def qdir(qnum): return ANNOTATIONS_DIR/f'q_{qnum}'
def list_annotation_files_for_qnum(qnum):
    d=qdir(qnum)
    return [] if not d.exists() else sorted(d.glob('*/*.json'))

def load_gold_for_qnum(qnum):
    files=[p for p in list_annotation_files_for_qnum(qnum) if p.name.startswith('annotation_')]
    gold=set(); first=None
    for p in files:
        obj=safe_load_json(p)
        if not obj: continue
        if first is None: first=obj
        for tr in (obj.get('triples') or []): gold.add(triple_to_tuple(tr))
    first=first or {}
    return {'qnum':qnum,'question':questions.get(qnum,''),'meta':{'mode':first.get('mode'),'aggregate':first.get('aggregate'),'intents':first.get('intents') or [],'scope':first.get('scope') or {},'exclude_scope':first.get('exclude_scope') or {},'doc_types':first.get('doc_types') or [],'topic_keys':first.get('topic_keys') or [],'biometric_modalities':first.get('biometric_modalities') or [],'conditional':first.get('conditional')},'gold_triples':gold,'n_annotation_files':len(files)}

gold_index={q:load_gold_for_qnum(q) for q in SELECTED_QNUMS}
pd.DataFrame([{'qnum':q,'mode':gold_index[q]['meta'].get('mode'),'gold_triplets':len(gold_index[q]['gold_triples']),'question':gold_index[q]['question']} for q in SELECTED_QNUMS]).sort_values('qnum')


,qnum,mode,gold_triplets,question
0,1,single,678,Which documents do I need to bring to passport...


In [22]:
def ensure_dict(x):
    return x if isinstance(x, dict) else {}

def normalize_surface(s): return re.sub(r'\s+',' ',s.lower().strip().replace('_',' '))
def key_to_surface(key):
    parts=key.split('_',1); return normalize_surface(parts[1] if len(parts)==2 else key)

entity_lexicon=defaultdict(set)
for q in SELECTED_QNUMS:
    meta=ensure_dict(gold_index[q].get('meta'))
    scope0=ensure_dict(meta.get('scope'))
    for bucket in ('cities','countries','airports'):
        for k in (ensure_dict(scope0).get(bucket,[]) or []):
            ks=canonicalize_key(k)
            if ks: entity_lexicon[key_to_surface(ks)].add(ks)
for q in SELECTED_QNUMS:
    for (s,p,o,sl,ol) in gold_index[q]['gold_triples']:
        if s: entity_lexicon[key_to_surface(s)].add(s)
        if o: entity_lexicon[key_to_surface(o)].add(o)
AGG_KWS=['most','fewest','highest','how often','more common','popular','usually','count']
def classify_question(question,meta):
    meta=ensure_dict(meta)
    if meta.get('mode')=='aggregate': return 'aggregate'
    ql=question.lower(); return 'aggregate' if any(kw in ql for kw in AGG_KWS) else 'single'
def detect_answer_type(question,meta):
    ql=question.lower()
    if 'which documents' in ql: return 'DocumentInstance'
    if 'question topics' in ql or 'asked about' in ql: return 'QuestionTopic'
    if 'biometric' in ql or 'fingerprint' in ql or 'face photo' in ql: return 'BiometricCheck'
    if 'which city' in ql or 'in which city' in ql: return 'City'
    if 'which country' in ql or 'through which country' in ql: return 'Country'
    if 'which airport' in ql: return 'Airport'
    return 'Entity'
def entity_link(question,meta):
    meta=ensure_dict(meta)
    ql=normalize_surface(question); linked=[]
    for surface,keys in entity_lexicon.items():
        if re.search(rf'\b{re.escape(surface)}\b',ql):
            for k in keys: linked.append({'key':k,'confidence':0.95,'source':'lexicon'})
    scope=ensure_dict(meta.get('scope'))
    for bucket in ('cities','countries','airports'):
        for k in (scope.get(bucket,[]) or []):
            ks=canonicalize_key(k)
            if ks and ks not in {x['key'] for x in linked}: linked.append({'key':ks,'confidence':0.90,'source':'scope'})
    best={}
    for x in linked:
        k=x['key']
        if k not in best or x['confidence']>best[k]['confidence']: best[k]=x
    return sorted(best.values(), key=lambda z:z['confidence'], reverse=True)


def detect_targets(question, meta):
    meta = ensure_dict(meta)
    ql = question.lower()
    intents = [str(x).lower() for x in (meta.get('intents') or [])]
    has_doc = ('which documents' in ql) or any('document' in x for x in intents) or ('documentcheck' in ''.join(intents))
    has_q = ('question topics' in ql) or ('asked about' in ql) or any('question' in x for x in intents)
    has_bio = ('biometric' in ql) or ('fingerprint' in ql) or ('face photo' in ql) or any('biometric' in x for x in intents)

    targets = []
    if has_doc:
        targets.append('documents')
    if has_q:
        targets.append('questions')
    if has_bio:
        targets.append('biometrics')
    if not targets:
        targets = ['general']
    return targets



# Domain exceptions: airports that should be treated as valid for a city scope.
CITY_AIRPORT_EXCEPTIONS = {
    'city_paris': ['airport_bva', 'airport_cdg', 'airport_ory'],
}

def expand_scope_with_exceptions(meta):
    meta2 = dict(ensure_dict(meta))
    scope_raw = meta2.get('scope') if isinstance(meta2, dict) else {}
    scope = dict(scope_raw or {})
    cities = list(scope.get('cities') or [])
    airports = set(scope.get('airports') or [])
    for c in cities:
        for a in CITY_AIRPORT_EXCEPTIONS.get(c, []):
            airports.add(a)
    scope['airports'] = sorted(airports)
    meta2['scope'] = scope
    return meta2





In [23]:
PATH_TEMPLATES=[
    {'name':'city_to_country','cypher':'MATCH (c:City {key:$start_key})-[:locatedInCountry]->(co:Country) RETURN [{s_key:c.key,p:"locatedInCountry",o_key:co.key,s_label:"City",o_label:"Country"}] AS triples LIMIT $limit'},
    {'name':'airport_to_city','cypher':'MATCH (a:Airport {key:$start_key})-[:locatedInCity]->(c:City) RETURN [{s_key:a.key,p:"locatedInCity",o_key:c.key,s_label:"Airport",o_label:"City"}] AS triples LIMIT $limit'},
    {'name':'airport_to_country','cypher':'MATCH (a:Airport {key:$start_key})-[:locatedInCountry]->(co:Country) RETURN [{s_key:a.key,p:"locatedInCountry",o_key:co.key,s_label:"Airport",o_label:"Country"}] AS triples LIMIT $limit'},
    {'name':'city_to_encounter_to_doc','cypher':'MATCH (e:Encounter)-[:atCity]->(c:City {key:$start_key}) MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance) RETURN [{s_key:e.key,p:"atCity",o_key:c.key,s_label:"Encounter",o_label:"City"},{s_key:e.key,p:"hasDocumentCheck",o_key:dc.key,s_label:"Encounter",o_label:"DocumentCheck"},{s_key:dc.key,p:"requestedDocument",o_key:di.key,s_label:"DocumentCheck",o_label:"DocumentInstance"},{s_key:di.key,p:"documentType",o_key:di.documentType,s_label:"DocumentInstance",o_label:"Literal"}] AS triples LIMIT $limit'},
    {'name':'city_to_encounter_to_topic','cypher':'MATCH (e:Encounter)-[:atCity]->(c:City {key:$start_key}) MATCH (e)-[:hasQuestioning]->(q:Questioning) RETURN [{s_key:e.key,p:"atCity",o_key:c.key,s_label:"Encounter",o_label:"City"},{s_key:e.key,p:"hasQuestioning",o_key:q.key,s_label:"Encounter",o_label:"Questioning"},{s_key:q.key,p:"topicKey",o_key:q.topicKey,s_label:"Questioning",o_label:"Literal"}] AS triples LIMIT $limit'},
    {'name':'country_to_encounter_to_doc','cypher':'MATCH (e:Encounter)-[:atCountry]->(co:Country {key:$start_key}) MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance) RETURN [{s_key:e.key,p:"atCountry",o_key:co.key,s_label:"Encounter",o_label:"Country"},{s_key:e.key,p:"hasDocumentCheck",o_key:dc.key,s_label:"Encounter",o_label:"DocumentCheck"},{s_key:dc.key,p:"requestedDocument",o_key:di.key,s_label:"DocumentCheck",o_label:"DocumentInstance"},{s_key:di.key,p:"documentType",o_key:di.documentType,s_label:"DocumentInstance",o_label:"Literal"}] AS triples LIMIT $limit'},
    {'name':'country_to_encounter_to_topic','cypher':'MATCH (e:Encounter)-[:atCountry]->(co:Country {key:$start_key}) MATCH (e)-[:hasQuestioning]->(q:Questioning) RETURN [{s_key:e.key,p:"atCountry",o_key:co.key,s_label:"Encounter",o_label:"Country"},{s_key:e.key,p:"hasQuestioning",o_key:q.key,s_label:"Encounter",o_label:"Questioning"},{s_key:q.key,p:"topicKey",o_key:q.topicKey,s_label:"Questioning",o_label:"Literal"}] AS triples LIMIT $limit'}
]
ALLOWED_RELATIONS=['atCity','atCountry','atAirport','locatedInCity','locatedInCountry','hasDocumentCheck','requestedDocument','hasQuestioning']

@dataclass
class RetrievedPath:
    template_name:str
    triples:list
    score:float
    source:str

def _open_driver():
    if GraphDatabase is None: raise RuntimeError('neo4j driver unavailable')
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER,NEO4J_PASSWORD))

def _row_to_path_triples(row):
    out=[]
    triples = row.get('triples',[]) if isinstance(row, dict) else []
    for tr in (triples or []):
        if not isinstance(tr, dict):
            continue
        s=canonicalize_key(tr.get('s_key')); p=canonicalize_key(tr.get('p')); o=canonicalize_key(tr.get('o_key')); sl=canonicalize_key(tr.get('s_label')); ol=canonicalize_key(tr.get('o_label'))
        if s and p and o: out.append((s,p,o,sl,ol))
    return out

def bfs_query(max_hops=3):
    return f"""
MATCH p=(start {{key: $start_key}})-[r*1..{max_hops}]-(ans)
WHERE ALL(rel IN r WHERE type(rel) IN $allowed_relations)
WITH relationships(p) AS rels
UNWIND rels AS rel
WITH DISTINCT rel
RETURN collect({{
  s_key:startNode(rel).key,
  p:type(rel),
  o_key:endNode(rel).key,
  s_label:head(labels(startNode(rel))),
  o_label:head(labels(endNode(rel)))
}}) AS triples
LIMIT $limit
""".strip()

def scoped_encounter_query():
    return """
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
// Geo filter must be applied when any geo scope is present.
// Geo precedence:
// 1) if cities specified -> city scope (plus airports located in those cities)
// 2) else if airports specified -> only those airports
// 3) else if countries specified -> all encounters in those countries (incl. cities/airports in country)
WHERE CASE
    WHEN has_cities THEN (match_city OR match_airport)
    WHEN has_airports THEN match_airport
    WHEN has_countries THEN match_country
    ELSE true
END
WITH DISTINCT e,a,c,co,c2,co2,co3
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
WITH e,a,c,co,c2,co2,co3,dc,di,q
CALL (e, dc, di, q, a, c, co, c2, co2, co3) {
  WITH e,dc,di,q,a,c,co,c2,co2,co3
  RETURN [
    CASE WHEN e IS NOT NULL AND dc IS NOT NULL THEN {s_key:e.key,p:'hasDocumentCheck',o_key:dc.key,s_label:labels(e)[0],o_label:labels(dc)[0]} END,
    CASE WHEN dc IS NOT NULL AND di IS NOT NULL THEN {s_key:dc.key,p:'requestedDocument',o_key:di.key,s_label:labels(dc)[0],o_label:labels(di)[0]} END,
    CASE WHEN di IS NOT NULL AND di.documentType IS NOT NULL THEN {s_key:di.key,p:'documentType',o_key:di.documentType,s_label:labels(di)[0],o_label:'Literal'} END,
    CASE WHEN e IS NOT NULL AND q IS NOT NULL THEN {s_key:e.key,p:'hasQuestioning',o_key:q.key,s_label:labels(e)[0],o_label:labels(q)[0]} END,
    CASE WHEN q IS NOT NULL AND q.topicKey IS NOT NULL THEN {s_key:q.key,p:'topicKey',o_key:q.topicKey,s_label:labels(q)[0],o_label:'Literal'} END,
    CASE WHEN e IS NOT NULL AND c IS NOT NULL THEN {s_key:e.key,p:'atCity',o_key:c.key,s_label:labels(e)[0],o_label:labels(c)[0]} END,
    CASE WHEN e IS NOT NULL AND a IS NOT NULL THEN {s_key:e.key,p:'atAirport',o_key:a.key,s_label:labels(e)[0],o_label:labels(a)[0]} END,
    CASE WHEN e IS NOT NULL AND co IS NOT NULL THEN {s_key:e.key,p:'atCountry',o_key:co.key,s_label:labels(e)[0],o_label:labels(co)[0]} END,
    CASE WHEN a IS NOT NULL AND c2 IS NOT NULL THEN {s_key:a.key,p:'locatedInCity',o_key:c2.key,s_label:labels(a)[0],o_label:labels(c2)[0]} END,
    CASE WHEN c IS NOT NULL AND co3 IS NOT NULL THEN {s_key:c.key,p:'locatedInCountry',o_key:co3.key,s_label:labels(c)[0],o_label:labels(co3)[0]} END,
    CASE WHEN a IS NOT NULL AND co2 IS NOT NULL THEN {s_key:a.key,p:'locatedInCountry',o_key:co2.key,s_label:labels(a)[0],o_label:labels(co2)[0]} END
  ] AS triples_raw
}
WITH [t IN triples_raw WHERE t IS NOT NULL] AS triples
RETURN triples
LIMIT $limit
""".strip()

def retrieve_paths_for_anchor(tx,anchor_key,meta,question_class,answer_type,targets):
    out=[]

    # templates (no target-family exclusion)
    for tpl in PATH_TEMPLATES:
        rows=tx.run(tpl['cypher'], start_key=anchor_key, limit=MAX_PATHS_PER_TEMPLATE).data()
        for row in rows:
            tr=_row_to_path_triples(row)
            if tr:
                out.append(RetrievedPath(tpl['name'],tr,0.0,'template'))

    # scoped retrieval with strict geo filter, but include all relation families for eligible geo subgraph
    meta = meta if isinstance(meta, dict) else {}
    scope = meta.get('scope') if isinstance(meta.get('scope'), dict) else {}
    rows=tx.run(
        scoped_encounter_query(),
        airports=scope.get('airports',[]) or [],
        cities=scope.get('cities',[]) or [],
        countries=scope.get('countries',[]) or [],
        doc_types=[],
        topic_keys=[],
        want_docs=True,
        want_topics=True,
        include_documents=True,
        include_questions=True,
        limit=max(MAX_PATHS_PER_TEMPLATE, 800)
    ).data()
    for row in rows:
        tr=_row_to_path_triples(row)
        if tr:
            out.append(RetrievedPath('scoped_encounter',tr,0.0,'scoped'))

    # bfs as fallback for general only
    if (not out) and ('general' in targets):
        rows=tx.run(bfs_query(MAX_HOPS), start_key=anchor_key, allowed_relations=ALLOWED_RELATIONS, limit=MAX_BFS_PATHS).data()
        for row in rows:
            tr=_row_to_path_triples(row)
            if tr:
                out.append(RetrievedPath('fallback_bfs',tr,0.0,'bfs'))

    return out









def geo_triples_query():
    return """
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN has_cities THEN (match_city OR match_airport)
    WHEN has_airports THEN match_airport
    WHEN has_countries THEN match_country
    ELSE true
END
WITH DISTINCT e,a,c,co,c2,co2,co3
WITH [
  CASE WHEN e IS NOT NULL AND c IS NOT NULL THEN {s_key:e.key,p:'atCity',o_key:c.key,s_label:labels(e)[0],o_label:labels(c)[0]} END,
  CASE WHEN e IS NOT NULL AND a IS NOT NULL THEN {s_key:e.key,p:'atAirport',o_key:a.key,s_label:labels(e)[0],o_label:labels(a)[0]} END,
  CASE WHEN e IS NOT NULL AND co IS NOT NULL THEN {s_key:e.key,p:'atCountry',o_key:co.key,s_label:labels(e)[0],o_label:labels(co)[0]} END,
  CASE WHEN a IS NOT NULL AND c2 IS NOT NULL THEN {s_key:a.key,p:'locatedInCity',o_key:c2.key,s_label:labels(a)[0],o_label:labels(c2)[0]} END,
  CASE WHEN c IS NOT NULL AND co3 IS NOT NULL THEN {s_key:c.key,p:'locatedInCountry',o_key:co3.key,s_label:labels(c)[0],o_label:labels(co3)[0]} END,
  CASE WHEN a IS NOT NULL AND co2 IS NOT NULL THEN {s_key:a.key,p:'locatedInCountry',o_key:co2.key,s_label:labels(a)[0],o_label:labels(co2)[0]} END
] AS triples_raw
RETURN [t IN triples_raw WHERE t IS NOT NULL] AS triples
LIMIT $limit
""".strip()


def retrieve_scope_geo_triples(tx, meta, limit=5000):
    meta = meta if isinstance(meta, dict) else {}
    scope = meta.get('scope') if isinstance(meta.get('scope'), dict) else {}
    rows=tx.run(
        geo_triples_query(),
        airports=scope.get('airports',[]) or [],
        cities=scope.get('cities',[]) or [],
        countries=scope.get('countries',[]) or [],
        limit=limit,
    ).data()
    out=set()
    for row in rows:
        for tr in _row_to_path_triples(row):
            out.add(tr)
    return out





In [24]:
def path_score(path,linked,answer_type):
    lk={x['key']:x['confidence'] for x in linked}; el=lk.get(path.triples[0][0],0.0) if path.triples else 0.0
    lb=1.0/(1.0+max(0,len(path.triples)-1)); last=(path.triples[-1][4] if path.triples else '').lower(); at=0.5 if answer_type.lower() in last else 0.0; tb=0.35 if path.source=='template' else (0.2 if path.source=='scoped' else 0.03)
    return el+0.6*lb+at+tb
def rank_paths(paths,linked,answer_type):
    for p in paths: p.score=path_score(p,linked,answer_type)
    paths=sorted(paths,key=lambda x:x.score,reverse=True)
    # keep high-quality non-template paths only
    filtered=[p for p in paths if (p.source=='template') or (p.score>=0.58)]
    base=filtered if filtered else paths
    return base[:TOP_K_PATHS]
def flatten_retrieved_triples(paths):
    out=set()
    for p in paths:
        for tr in p.triples: out.add(tr)
    return out
def compute_metrics(gold,pred):
    tp=len(gold&pred); fp=len(pred-gold); fn=len(gold-pred)
    precision=tp/(tp+fp) if (tp+fp) else 0.0; recall=tp/(tp+fn) if (tp+fn) else 0.0; f1=(2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
    return {'tp':tp,'fp':fp,'fn':fn,'precision':precision,'recall':recall,'f1':f1}



In [25]:
results=[]; details=[]; driver=None; err=None
try: driver=_open_driver()
except Exception as e: err=str(e); print('neo4j connection failed:',err)
for qnum in SELECTED_QNUMS:
    obj=gold_index.get(qnum, {}); q=obj.get('question',''); meta_raw=obj.get('meta') if isinstance(obj, dict) else {}; meta=expand_scope_with_exceptions(meta_raw); gold=obj.get('gold_triples', set()); qclass=classify_question(q,meta); ans_type=detect_answer_type(q,meta); targets=detect_targets(q,meta); linked=entity_link(q,meta)
    if qclass=='aggregate' and len(gold)==0:
        m={'tp':0,'fp':0,'fn':0,'precision':0.0,'recall':0.0,'f1':0.0}
        results.append({'qnum':qnum,'question':q,'question_class':qclass,'answer_type':ans_type,'n_linked_entities':len(linked),'n_gold_triples':0,'n_retrieved_triples':0,'skipped_aggregate_no_gold':1,'driver_ok':0 if driver is None else 1,**m,'error':''}); continue
    if driver is None:
        m={'tp':0,'fp':0,'fn':len(gold),'precision':0.0,'recall':0.0,'f1':0.0}
        results.append({'qnum':qnum,'question':q,'question_class':qclass,'answer_type':ans_type,'n_linked_entities':len(linked),'n_gold_triples':len(gold),'n_retrieved_triples':0,'skipped_aggregate_no_gold':0,'driver_ok':0,**m,'error':f'neo4j_unavailable: {err}'}); continue
    try:
        paths=[]
        with driver.session() as session:
            for ent in linked: paths.extend(session.execute_read(retrieve_paths_for_anchor, ent['key'], meta, qclass, ans_type, targets))
        n_paths_total=len(paths); ranked=rank_paths(paths,linked,ans_type); pred=flatten_retrieved_triples(ranked)
        # Always include geo triples for eligible scope encounters
        with driver.session() as geo_sess:
            geo_tr=geo_sess.execute_read(retrieve_scope_geo_triples, meta)
        pred = pred | geo_tr
        m=compute_metrics(gold,pred)
        results.append({'qnum':qnum,'question':q,'question_class':qclass,'answer_type':ans_type,'n_linked_entities':len(linked),'linked_entities':json.dumps(linked,ensure_ascii=False),'n_gold_triples':len(gold),'n_retrieved_triples':len(pred),'n_paths_total':n_paths_total,'n_paths_ranked':len(ranked),'skipped_aggregate_no_gold':0,'driver_ok':1,**m,'error':''})
        for i,p in enumerate(ranked,start=1): details.append({'qnum':qnum,'rank':i,'template':p.template_name,'source':p.source,'score':p.score,'path_json':json.dumps([[s,pr,o] for (s,pr,o,sl,ol) in p.triples],ensure_ascii=False)})
    except Exception as e:
        m={'tp':0,'fp':0,'fn':len(gold),'precision':0.0,'recall':0.0,'f1':0.0}
        results.append({'qnum':qnum,'question':q,'question_class':qclass,'answer_type':ans_type,'n_linked_entities':len(linked),'n_gold_triples':len(gold),'n_retrieved_triples':0,'skipped_aggregate_no_gold':0,'driver_ok':1,**m,'error':str(e)})
if driver is not None: driver.close()
metrics_df=pd.DataFrame(results).sort_values('qnum'); details_df=pd.DataFrame(details).sort_values(['qnum','rank']) if details else pd.DataFrame()
metrics_df.to_csv(CSV_OUT,index=False,encoding='utf-8'); details_df.to_csv(DETAIL_CSV_OUT,index=False,encoding='utf-8')
print('saved',CSV_OUT); print('saved',DETAIL_CSV_OUT)
metrics_df[['qnum','question_class','n_gold_triples','n_retrieved_triples','precision','recall','f1','skipped_aggregate_no_gold','error']]









saved /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/graphrag_triplet_retrieval_metrics.csv
saved /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/graphrag_triplet_retrieval_details.csv


,qnum,question_class,n_gold_triples,n_retrieved_triples,precision,recall,f1,skipped_aggregate_no_gold,error
0,1,single,678,744,0.91129,1.0,0.953586,0,


In [26]:
if len(metrics_df):
    summary={'n_questions':int(len(metrics_df)),'n_skipped_aggregate_no_gold':int(metrics_df['skipped_aggregate_no_gold'].sum()),'macro_precision':float(metrics_df['precision'].mean()),'macro_recall':float(metrics_df['recall'].mean()),'macro_f1':float(metrics_df['f1'].mean())}
    print(json.dumps(summary,indent=2,ensure_ascii=False))


{
  "n_questions": 1,
  "n_skipped_aggregate_no_gold": 0,
  "macro_precision": 0.9112903225806451,
  "macro_recall": 1.0,
  "macro_f1": 0.9535864978902954
}


In [27]:
pred

{('airport_bva', 'locatedincity', 'city_beauvais', 'airport', 'city'),
 ('airport_cdg', 'locatedincity', 'city_paris', 'airport', 'city'),
 ('airport_ory', 'locatedincity', 'city_paris', 'airport', 'city'),
 ('city_beauvais', 'locatedincountry', 'country_france', 'city', 'country'),
 ('city_paris', 'locatedincountry', 'country_france', 'city', 'country'),
 ('doccheck_encounter_10918979_author_2_foreign_passport',
  'requesteddocument',
  'documentinstance_encounter_10918979_author_2_foreign_passport',
  'documentcheck',
  'documentinstance'),
 ('doccheck_encounter_10918979_author_2_hotel_booking_doc',
  'requesteddocument',
  'documentinstance_encounter_10918979_author_2_hotel_booking_doc',
  'documentcheck',
  'documentinstance'),
 ('doccheck_encounter_11326744_author_1_foreign_passport',
  'requesteddocument',
  'documentinstance_encounter_11326744_author_1_foreign_passport',
  'documentcheck',
  'documentinstance'),
 ('doccheck_encounter_11383255_author_1_foreign_passport',
  'reque

In [113]:
# Single-question F1 optimization (without changing output format)
# Default target is the hardest aggregate question; change if needed.
TARGET_QNUM = 1

# Candidate grid (adjust if search is too slow)
GRID_TOP_K = [40, 80, 120, 200]
GRID_MAX_TEMPLATE = [120, 240, 400]
GRID_MAX_BFS = [0, 40, 120]
GRID_MIN_NON_TEMPLATE_SCORE = [0.52, 0.58, 0.64, 0.70]
GRID_TEMPLATE_BONUS = [0.30, 0.35, 0.42]
GRID_SCOPED_BONUS = [0.16, 0.20, 0.26]
GRID_BFS_BONUS = [0.00, 0.03]



In [114]:
from itertools import product

def rank_paths_param(paths, linked, answer_type, top_k, min_non_template_score, template_bonus, scoped_bonus, bfs_bonus):
    lk={x['key']:x['confidence'] for x in linked}
    scored=[]
    for p in paths:
        el=lk.get(p.triples[0][0],0.0) if p.triples else 0.0
        lb=1.0/(1.0+max(0,len(p.triples)-1))
        last=(p.triples[-1][4] if p.triples else '').lower()
        at=0.5 if answer_type.lower() in last else 0.0
        tb=template_bonus if p.source=='template' else (scoped_bonus if p.source=='scoped' else bfs_bonus)
        p.score=el+0.6*lb+at+tb
        scored.append(p)
    scored=sorted(scored,key=lambda x:x.score,reverse=True)
    filtered=[p for p in scored if (p.source=='template') or (p.score>=min_non_template_score)]
    base=filtered if filtered else scored
    return base[:top_k]

def evaluate_single_q_params(qnum, top_k, max_template, max_bfs, min_non_template_score, template_bonus, scoped_bonus, bfs_bonus):
    global MAX_PATHS_PER_TEMPLATE, MAX_BFS_PATHS
    obj=gold_index[qnum]
    q=obj['question']; meta_raw=obj.get('meta') if isinstance(obj, dict) else {}; meta=expand_scope_with_exceptions(meta_raw); gold=obj['gold_triples']
    qclass=classify_question(q,meta); ans_type=detect_answer_type(q,meta); targets=detect_targets(q,meta); linked=entity_link(q,meta)

    if qclass=='aggregate' and len(gold)==0:
        return {'f1':0.0,'precision':0.0,'recall':0.0,'n_retrieved':0,'n_paths_total':0,'skip':1,'error':''}

    old_tpl, old_bfs = MAX_PATHS_PER_TEMPLATE, MAX_BFS_PATHS
    MAX_PATHS_PER_TEMPLATE, MAX_BFS_PATHS = max_template, max_bfs
    try:
        driver=_open_driver()
        try:
            paths=[]
            with driver.session() as session:
                for ent in linked:
                    paths.extend(session.execute_read(retrieve_paths_for_anchor, ent['key'], meta, qclass, ans_type, targets))
        finally:
            driver.close()
    except Exception as e:
        MAX_PATHS_PER_TEMPLATE, MAX_BFS_PATHS = old_tpl, old_bfs
        return {'f1':0.0,'precision':0.0,'recall':0.0,'n_retrieved':0,'n_paths_total':0,'skip':0,'error':str(e)}

    ranked=rank_paths_param(paths, linked, ans_type, top_k, min_non_template_score, template_bonus, scoped_bonus, bfs_bonus)
    pred=flatten_retrieved_triples(ranked)
    try:
        drv=_open_driver()
        try:
            with drv.session() as geo_sess:
                geo_tr=geo_sess.execute_read(retrieve_scope_geo_triples, meta)
            pred = pred | geo_tr
        finally:
            drv.close()
    except Exception:
        pass
    m=compute_metrics(gold,pred)
    MAX_PATHS_PER_TEMPLATE, MAX_BFS_PATHS = old_tpl, old_bfs
    return {**m,'n_retrieved':len(pred),'n_paths_total':len(paths),'skip':0,'error':''}

search_rows=[]
for top_k, max_tpl, max_bfs, min_score, tpl_b, sc_b, bfs_b in product(
    GRID_TOP_K, GRID_MAX_TEMPLATE, GRID_MAX_BFS, GRID_MIN_NON_TEMPLATE_SCORE,
    GRID_TEMPLATE_BONUS, GRID_SCOPED_BONUS, GRID_BFS_BONUS
):
    r=evaluate_single_q_params(
        TARGET_QNUM,
        top_k=top_k,
        max_template=max_tpl,
        max_bfs=max_bfs,
        min_non_template_score=min_score,
        template_bonus=tpl_b,
        scoped_bonus=sc_b,
        bfs_bonus=bfs_b,
    )
    search_rows.append({
        'qnum':TARGET_QNUM,
        'top_k':top_k,
        'max_template':max_tpl,
        'max_bfs':max_bfs,
        'min_non_template_score':min_score,
        'template_bonus':tpl_b,
        'scoped_bonus':sc_b,
        'bfs_bonus':bfs_b,
        **r,
    })

search_df=pd.DataFrame(search_rows)
best_df=search_df.sort_values(['f1','recall','precision'],ascending=False)
print('Top configs:')
display(best_df.head(10))

BEST = best_df.iloc[0].to_dict()
print('BEST=', {k: BEST[k] for k in ['top_k','max_template','max_bfs','min_non_template_score','template_bonus','scoped_bonus','bfs_bonus','precision','recall','f1','n_retrieved']})








Top configs:


,qnum,top_k,max_template,max_bfs,min_non_template_score,template_bonus,scoped_bonus,bfs_bonus,tp,fp,fn,precision,recall,f1,n_retrieved,n_paths_total,skip,error
2160,1,200,240,0,0.52,0.30,0.16,0.00,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2161,1,200,240,0,0.52,0.30,0.16,0.03,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2162,1,200,240,0,0.52,0.30,0.20,0.00,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2163,1,200,240,0,0.52,0.30,0.20,0.03,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2164,1,200,240,0,0.52,0.30,0.26,0.00,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2165,1,200,240,0,0.52,0.30,0.26,0.03,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2166,1,200,240,0,0.52,0.35,0.16,0.00,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2167,1,200,240,0,0.52,0.35,0.16,0.03,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2168,1,200,240,0,0.52,0.35,0.20,0.00,526,66,152,0.888514,0.775811,0.828346,592,985,0,
2169,1,200,240,0,0.52,0.35,0.20,0.03,526,66,152,0.888514,0.775811,0.828346,592,985,0,


BEST= {'top_k': 200, 'max_template': 240, 'max_bfs': 0, 'min_non_template_score': 0.52, 'template_bonus': 0.3, 'scoped_bonus': 0.16, 'bfs_bonus': 0.0, 'precision': 0.8885135135135135, 'recall': 0.775811209439528, 'f1': 0.8283464566929133, 'n_retrieved': 592}


In [115]:
# Apply best params found above to the main run config
# Then rerun the main run cell to regenerate CSV in the same format.
TOP_K_PATHS = int(BEST['top_k'])
MAX_PATHS_PER_TEMPLATE = int(BEST['max_template'])
MAX_BFS_PATHS = int(BEST['max_bfs'])

# Rebind ranking functions with tuned constants
def path_score(path,linked,answer_type):
    lk={x['key']:x['confidence'] for x in linked}
    el=lk.get(path.triples[0][0],0.0) if path.triples else 0.0
    lb=1.0/(1.0+max(0,len(path.triples)-1))
    last=(path.triples[-1][4] if path.triples else '').lower()
    at=0.5 if answer_type.lower() in last else 0.0
    tb=float(BEST['template_bonus']) if path.source=='template' else (float(BEST['scoped_bonus']) if path.source=='scoped' else float(BEST['bfs_bonus']))
    return el+0.6*lb+at+tb

def rank_paths(paths,linked,answer_type):
    for p in paths:
        p.score=path_score(p,linked,answer_type)
    paths=sorted(paths,key=lambda x:x.score,reverse=True)
    filtered=[p for p in paths if (p.source=='template') or (p.score>=float(BEST['min_non_template_score']))]
    base=filtered if filtered else paths
    return base[:TOP_K_PATHS]

print('Applied tuned params for qnum', TARGET_QNUM)
print('TOP_K_PATHS=', TOP_K_PATHS, 'MAX_PATHS_PER_TEMPLATE=', MAX_PATHS_PER_TEMPLATE, 'MAX_BFS_PATHS=', MAX_BFS_PATHS)



Applied tuned params for qnum 1
TOP_K_PATHS= 200 MAX_PATHS_PER_TEMPLATE= 240 MAX_BFS_PATHS= 0


In [116]:
pred

{('documentinstance_encounter_11794216_author_1_visa',
  'documenttype',
  'visa',
  'documentinstance',
  'literal'),
 ('encounter_11790790_friend_1',
  'hasdocumentcheck',
  'doccheck_encounter_11790790_friend_1_hotel_booking',
  'encounter',
  'documentcheck'),
 ('encounter_11869687_author_1',
  'hasdocumentcheck',
  'doccheck_encounter_11869687_author_1_return_ticket',
  'encounter',
  'documentcheck'),
 ('documentinstance_encounter_11520663_author_1_return_ticket',
  'documenttype',
  'return_ticket',
  'documentinstance',
  'literal'),
 ('encounter_11927366_author_1',
  'hasdocumentcheck',
  'doccheck_encounter_11927366_author_1_foreign_passport',
  'encounter',
  'documentcheck'),
 ('doccheck_encounter_11476326_author_1_foreign_passport',
  'requesteddocument',
  'documentinstance_encounter_11476326_author_1_foreign_passport',
  'documentcheck',
  'documentinstance'),
 ('documentinstance_encounter_11455451_author_1_foreign_passport',
  'documenttype',
  'foreign_passport',
  'do